# 2.5. Automatic Differentiation
D2L의 Automatic Differentiation 장을 PyTorch 기준으로 정리함.
딥러닝에서 중요한 계산은 결국 `loss계산` -> `loss를 각 파라미터로 미분` -> `gradient방향 보고 파라미터 업데이트` 이다.

그런데 모델이 복잡해지면 사람이 직접 미분식을 계산하는 것은 비효율적이고 실수도 많다.
그래서 PyTorch 같은 딥러닝 프레임워크는 자동 미분이 있다.
자동 미분은 어떤 값이 어떤 값으로부터 계산됐는지를 계산 그래프로 기록한다. 
그리고 .backward()를 호출하면 그 그래프를 거꾸로 따라가며 chain rule로 gradient를 계산한다. 

## 0. 기본 설정

PyTorch를 불러오고 현재 환경을 확인

In [1]:
import torch
print("PyTorch version:", torch.__version__)

PyTorch version: 2.11.0+cu128


## 1. requires_grad

PyTorch에서 어떤 Tensor의 미분값을 구하려면 requires_grad=True를 설정해야 한다.

requires_grad=True
= 이 Tensor를 이용해 계산한 결과에 대해 나중에 gradient를 구하겠다.

먼저 간단한 Tensor를 만든다.

In [2]:
x = torch.arange(4.0)
print(x)

tensor([0., 1., 2., 3.])


처음 만든 x는 아직 gradient 추적 대상이 아니다.

이제 requires_grad_(True)를 사용해서 x의 gradient를 추적하도록 설정한다.

In [3]:
x.requires_grad_(True) 
print(x) 
print(x.grad)

tensor([0., 1., 2., 3.], requires_grad=True)
None


아직 backward()를 하지 않았기 때문에 x.grad는 None이다.

즉, gradient는 자동으로 생기는 것이 아니라, backward()를 실행해야 계산된다.

## 2. 간단한 함수 미분하기

D2L 함수 예제는 다음 함수이다.

$$
y = 2 x^Tx
$$

여기서 xᵀx는 벡터 x의 각 원소를 제곱해서 모두 더한 값이다.

예를 들어 x = [0, 1, 2, 3]이면 다음과 같다.

$$
x = [0, 1, 2, 3]
xᵀx = 0² + 1² + 2² + 3²
     = 0 + 1 + 4 + 9
     = 14
y = 2 * xᵀx
  = 2 * 14
  = 28
$$

수학적으로 이 함수의 미분은 다음과 같다.

$$
y = 2 * xᵀx \\
dy/dx = 4x
$$

`torch.dot(x, x)`는 각 원소를 곱한 뒤 전부 더해서 스칼라 하나를 만든다.
반면 `x * x`는 각 원소를 제곱한 결과를 그대로 유지하므로 벡터가 남는다.

PyTorch가 이 gradient를 자동으로 계산하는지 확인한다.

In [4]:
y = 2 * torch.dot(x, x) # dot은 내적 계산
print(y)

tensor(28., grad_fn=<MulBackward0>)


y는 x를 이용해서 계산된 값이다.

이제 y.backward()를 실행하면 PyTorch가 y를 x에 대해 미분한다.

In [5]:
y.backward()
print(x.grad)

tensor([ 0.,  4.,  8., 12.])


결과는 다음과 같다.

x = [0, 1, 2, 3]
4x = [0, 4, 8, 12]

따라서 x.grad에는 dy/dx = 4x가 저장된다.

자동 미분은 사람이 직접 미분식을 계산하지 않아도 PyTorch가 계산 그래프를 따라 gradient를 구해주는 기능이다.

In [6]:
print(x.grad == 4 * x)

tensor([True, True, True, True])


## 3. gradient는 누적된다

PyTorch에서 gradient는 자동으로 초기화되지 않는다.
backward()를 여러 번 실행하면 gradient가 기존 값에 더해질 수 있다.
그래서 새로운 미분을 계산하기 전에는 gradient를 0으로 초기화해야 한다.

In [7]:
x.grad.zero_()
print(x.grad)

tensor([0., 0., 0., 0.])


이제 다른 함수를 미분해본다.
$$
y = x₁ + x₂ + x₃ + x₄
$$
이 함수는 각 x에 대해 미분하면 모두 1이다.

In [8]:
y = x.sum()
y.backward()

print(x.grad)

tensor([1., 1., 1., 1.])


결과가 모두 1인 이유는 다음과 같다.
$$
y = x₁ + x₂ + x₃ + x₄\\
dy/dx₁ = 1\\
dy/dx₂ = 1\\
dy/dx₃ = 1\\
dy/dx₄ = 1\\
$$

딥러닝 학습 코드에서 자주 보는 다음 구조도 같은 흐름이다.

optimizer.zero_grad() # 이전 gradient 삭제  
loss.backward() # 현재 loss 기준으로 gradient 계산  
optimizer.step() # 계산된 gradient로 파라미터 이동  

즉 `loss.backward()`는 미분을 계산하는 단계이고, `optimizer.step()`은 그 미분값을 써서 실제로 `w`, `b` 같은 파라미터를 바꾸는 단계다.

## 4. 출력이 벡터일 때

지금까지는 출력 y가 숫자 하나였다.

하지만 다음처럼 계산하면 y는 벡터가 된다.
$$
y = x * x
$$
이 경우 결과는 다음과 같다.
$$
y = [x₁², x₂², x₃², x₄²]
$$
출력이 여러 개이면 y.backward()를 바로 실행할 수 없다.

PyTorch 입장에서는 여러 출력 중 무엇을 시작점으로 역전파해야 하는지 바로 정해지지 않기 때문이다.

여기서 `x * x`는 원소별 곱이라서 결과가 벡터로 남고, 앞에서 본 `torch.dot(x, x)`는 합까지 끝낸 스칼라라는 점이 핵심이다.

In [9]:
x.grad.zero_()
y = x * x
print(y)

tensor([0., 1., 4., 9.], grad_fn=<MulBackward0>)


출력이 벡터이므로 보통 하나의 스칼라 값으로 합친 뒤 미분한다.

여기서는 `y.sum()`을 사용한다.
`y` 안의 값 4개를 하나로 묶어서, 무엇을 기준으로 gradient를 구할지 분명하게 만드는 셈이다.

$$y.sum() = x₁² + x₂² + x₃² + x₄²
$$
이 값을 x에 대해 미분하면 다음과 같다.

$$d(y.sum())/dx = 2x$$

In [10]:
y.sum().backward()
print(x.grad)

tensor([0., 2., 4., 6.])


결과는 다음과 같다.

$$x = [0, 1, 2, 3]
2x = [0, 2, 4, 6]$$

정리하면 다음과 같다.

`torch.dot(x, x)`처럼 출력이 숫자 하나이면 바로 `y.backward()`를 호출할 수 있다.

`x * x`처럼 출력이 벡터나 행렬이면 보통 `y.sum().backward()`나 `y.mean().backward()`처럼 먼저 하나의 스칼라로 만든다.

딥러닝에서 loss를 보통 하나의 스칼라 값으로 만드는 이유도 여기에 있다.

## 5. detach

detach()는 Tensor를 계산 그래프에서 떼어내는 기능이다.

값은 그대로 사용하지만,
gradient 연결은 여기서 끊고 이 값이 어떻게 계산되었는지는 더 이상 추적하지 않는다.

먼저 y = x * x를 계산한다.

그다음 u = y.detach()를 만들면 u는 값은 y와 같지만, PyTorch는 u가 x로부터 계산되었다는 사실을 추적하지 않는다.

In [11]:
x.grad.zero_() 
y = x * x 
u = y.detach() 
z = u * x 
z.sum().backward() 
print("x:", x) 
print("y:", y) 
print("u:", u) 
print("x.grad:", x.grad)

x: tensor([0., 1., 2., 3.], requires_grad=True)
y: tensor([0., 1., 4., 9.], grad_fn=<MulBackward0>)
u: tensor([0., 1., 4., 9.])
x.grad: tensor([0., 1., 4., 9.])


여기서 중요한 부분은 다음이다.

$$u = y.detach()$$

u는 값만 보면 x²이다.

하지만 detach()를 했기 때문에 PyTorch는 u를 x와 연결된 값으로 보지 않는다.

$$z = u * x$$

PyTorch는 u를 상수처럼 취급한다.

그래서 미분 결과는 다음과 같다.

$$dz/dx = u$$

x.grad는 u와 같아진다.

In [12]:
print(x.grad == u)

tensor([True, True, True, True])


detach()는 값을 사용하되 gradient는 그 앞쪽으로 보내고 싶지 않을 때 사용한다.

초반에는 자주 쓰지 않지만, 나중에 모델 학습 구조가 복잡해지면 중요해진다.

## 6. 자동 미분과 딥러닝 학습 연결

딥러닝 학습에서는 보통 다음 흐름을 사용한다.

예측값 = 모델(x)  
loss = 예측값과 정답의 차이  
loss.backward()  

`loss.backward()`를 실행하면 PyTorch가 계산 그래프를 거꾸로 따라가며 chain rule로 loss를 각 파라미터에 대해 미분한다.

예를 들어 모델 안에 w, b가 있으면 다음 값들이 계산된다.

$$w.grad = ∂Loss/∂w  \\
b.grad = ∂Loss/∂b$$

`w`와 `b`는 모델이 학습하는 파라미터다. 보통 `w`는 가중치, `b`는 bias를 뜻한다.
`w.grad`는 loss를 `w`에 대해 미분한 값이고, `b.grad`는 loss를 `b`에 대해 미분한 값이다.
즉 gradient는 파라미터를 조금 바꿨을 때 loss가 얼마나 변하는지 보여주는 기울기 정보다.

그다음 optimizer는 이 gradient를 이용해 파라미터를 업데이트한다.

$$w = w - learning\_rate * w.grad\\
b = b - learning\_rate * b.grad$$

`learning_rate`는 모델 가중치 자체가 아니라, 한 번 업데이트할 때 얼마나 멀리 움직일지 정하는 보폭이다.
gradient가 양수이면 파라미터를 줄이고, gradient가 음수이면 파라미터를 늘린다.
이렇게 하는 이유는 loss가 증가하는 방향이 아니라, loss가 감소하는 방향으로 파라미터를 이동시키기 위해서다.

## 7. 오늘의 정리
- 자동 미분은 PyTorch가 gradient를 자동으로 계산해주는 기능이다.
- requires_grad=True를 설정하면 해당 Tensor의 gradient를 추적할 수 있다.
- backward()를 호출하면 계산 그래프를 거꾸로 따라가며 chain rule로 미분한다.
- 미분 결과는 .grad에 저장된다.
- y = 2 * xᵀx의 gradient는 4x다.
- PyTorch의 gradient는 자동으로 초기화되지 않고 누적된다.
- 그래서 반복 계산 전에는 zero_grad() 또는 x.grad.zero_()가 필요하다.
- `torch.dot(x, x)`는 스칼라를 만들고, `x * x`는 원소별 계산 결과인 벡터를 남긴다.
- 출력이 벡터나 행렬이면 바로 backward() 할 수 없고, 이때는 `y.sum().backward()`처럼 하나의 스칼라 값으로 바꾼 뒤 미분한다.
- detach()는 값은 사용하지만 계산 그래프 연결은 끊는 기능이다.
- 딥러닝 학습에서는 `loss.backward()`가 gradient를 계산하고 `optimizer.step()`이 파라미터를 업데이트한다.
- `learning_rate`는 파라미터 자체가 아니라 업데이트 보폭이다.